# SARR ETL — BigQuery → Embed (GPU) → Qdrant Cloud

1. Set runtime to **GPU**.
2. Fill secrets / env in the config cell.
3. For first load set `LAST_UPDATE_DATE = "1970-01-01"`.
4. Later runs: set watermark to the previous `max(update_date)`.

In [ ]:
# 1) Runtime → Change runtime type → T4 GPU (or better)
# 2) Get the code into Colab (pick ONE)

# Option A — clone from GitHub (private repos need a token)
# !git clone https://github.com/YOUR_ORG/sarr-recommendation-api.git
# %cd sarr-recommendation-api

# Option B — upload the repo zip via Colab Files, then:
# !unzip -q sarr-recommendation-api.zip
# %cd sarr-recommendation-api

!pip install -q -e ".[etl]"

In [ ]:
import os
from pathlib import Path

# --- Config (edit these) ---
os.environ["GCP_PROJECT_ID"] = "YOUR_GCP_PROJECT"
os.environ["BQ_DATASET"] = "YOUR_DATASET"
os.environ["BQ_TABLE"] = "YOUR_ENRICHED_PYPI_TABLE"
os.environ["QDRANT_URL"] = "https://YOUR-CLUSTER.aws.cloud.qdrant.io"
os.environ["QDRANT_API_KEY"] = "YOUR_QDRANT_KEY"
os.environ["QDRANT_COLLECTION"] = "sarr_pypi"
os.environ["EMBEDDING_MODEL"] = "BAAI/bge-small-en-v1.5"
os.environ["EMBEDDING_DIM"] = "384"

# First full load:
os.environ["LAST_UPDATE_DATE"] = "1970-01-01"
# Later incremental runs — set to previous max update_date, e.g.:
# os.environ["LAST_UPDATE_DATE"] = "2026-07-01"

# --- BigQuery auth (pick ONE) ---

# Option A: upload a GCP service-account JSON in Colab Files, then:
# SA_PATH = "/content/gcp-sa.json"
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = SA_PATH

# Option B: interactive user login in Colab
from google.colab import auth
auth.authenticate_user()

print("Config ready. Watermark =", os.environ["LAST_UPDATE_DATE"])
Path("data").mkdir(exist_ok=True)

In [ ]:
import torch
from sarr.common.config import get_settings
from sarr.etl.embed import BatchEmbedder
from sarr.etl.pipeline import run_etl

assert torch.cuda.is_available(), "Enable a GPU runtime before running ETL"

get_settings.cache_clear()
settings = get_settings()
print("BQ table:", f"{settings.gcp_project_id}.{settings.bq_dataset}.{settings.bq_table}")
print("Qdrant:", settings.qdrant_url, "collection=", settings.qdrant_collection)
print("Embed model:", settings.embedding_model)

embedder = BatchEmbedder(settings.embedding_model, device="cuda")

stats = run_etl(
    last_update_date=settings.last_update_date,
    batch_size=64,  # lower to 32 if you hit GPU OOM
    watermark_path="data/last_update_date.txt",
    settings=settings,
    embedder=embedder,
)
print(stats)
print("New watermark saved to data/last_update_date.txt — reuse it next month")